# 📝 เฉลย: Agentic RAG Workshop (4 ชม.)
## Agentic RAG: From Zero to Hero

---

> ⚠️ **ไฟล์นี้เป็นเฉลยสำหรับอาจารย์เท่านั้น — ห้ามแจกนักศึกษา**

### 📋 เนื้อหา
- ขั้นตอนที่ 1: Chunk + Embed + Search (3 คะแนน) — เฉลยเต็ม
- ขั้นตอนที่ 2: Agent + Custom Tool (3 คะแนน) — ตัวอย่างเฉลย (GPA calculator)
- ขั้นตอนที่ 3: RAG Agent + วัดคุณภาพ (4 คะแนน) — เฉลยเต็ม

## 📦 ติดตั้ง Dependencies

In [ ]:
%%time
!pip install -q google-adk google-genai sentence-transformers qdrant-client langchain-text-splitters scikit-learn

import hashlib, os, json, random, numpy as np, re
from sklearn.metrics.pairwise import cosine_similarity
print('✅ ติดตั้งเรียบร้อย!')

## 🎓 กรอกข้อมูลนักศึกษา (ใช้ข้อมูลตัวอย่างสำหรับเฉลย)

In [ ]:
# ─── ข้อมูลตัวอย่างสำหรับเฉลย ───
STUDENT_NAME = 'อาจารย์ เฉลย'   # ตัวอย่าง
STUDENT_ID   = '9999999999'     # ตัวอย่าง
PHONE        = '000-000-0000'
LINE_ID      = 'teacher_key'

# ─── ตรวจสอบ ───
assert len(STUDENT_ID) >= 5, '❌ กรุณากรอกรหัสนักศึกษา!'
assert len(STUDENT_NAME) >= 2, '❌ กรุณากรอกชื่อ-นามสกุล!'

print(f'✅ ชื่อ-นามสกุล: {STUDENT_NAME}')
print(f'✅ รหัสนักศึกษา: {STUDENT_ID}')
print(f'📱 เบอร์โทร: {PHONE}')
print(f'💬 LINE ID: {LINE_ID}')

## 📄 สร้างชุดข้อมูลเฉพาะตัว (ห้ามแก้ไข cell นี้)

In [ ]:
%%time
# ===== ห้ามแก้ไข cell นี้ =====
# สร้างชุดข้อมูลเฉพาะจากรหัสนักศึกษา

random.seed(int(hashlib.md5(STUDENT_ID.encode()).hexdigest()[:8], 16))

all_paragraphs = [
    'การเรียนรู้ของเครื่อง หรือ Machine Learning เป็นสาขาย่อยของปัญญาประดิษฐ์ที่มุ่งเน้นการพัฒนาอัลกอริทึมที่สามารถเรียนรู้จากข้อมูลและปรับปรุงประสิทธิภาพได้โดยอัตโนมัติ',
    'Deep Learning เป็นเทคนิคของ Machine Learning ที่ใช้โครงข่ายประสาทเทียมหลายชั้น Neural Network ในการประมวลผลข้อมูลที่ซับซ้อน เช่น การจดจำภาพ การแปลภาษา',
    'Natural Language Processing หรือ NLP คือสาขาที่ทำให้คอมพิวเตอร์สามารถเข้าใจ ตีความ และสร้างภาษามนุษย์ได้ รวมถึงการวิเคราะห์อารมณ์และการสรุปข้อความ',
    'Retrieval Augmented Generation หรือ RAG เป็นเทคนิคที่รวมการค้นหาข้อมูลเข้ากับการสร้างข้อความของ LLM เพื่อให้ได้คำตอบที่ถูกต้องและอ้างอิงแหล่งข้อมูลได้',
    'Vector Database เป็นฐานข้อมูลที่ออกแบบมาเพื่อจัดเก็บและค้นหาข้อมูลในรูปแบบ Embedding Vector ช่วยให้ค้นหาข้อมูลที่มีความหมายคล้ายกันได้รวดเร็ว',
    'Text Embedding คือกระบวนการแปลงข้อความให้เป็นชุดตัวเลข Vector ที่แสดงความหมายเชิงความหมายของข้อความนั้นได้ ทำให้เปรียบเทียบความคล้ายระหว่างข้อความได้',
    'Transformer เป็นสถาปัตยกรรมของ Neural Network ที่ใช้กลไก Attention ในการประมวลผลข้อมูล เป็นพื้นฐานของ GPT BERT และ Gemini',
    'Prompt Engineering คือศาสตร์ของการออกแบบคำสั่ง Prompt ที่ให้กับ LLM เพื่อให้ได้ผลลัพธ์ที่ต้องการ การเขียน Prompt ที่ดีช่วยเพิ่มคุณภาพคำตอบอย่างมาก',
    'Chunking คือกระบวนการแบ่งเอกสารขนาดยาวออกเป็นส่วนย่อยที่เหมาะสมสำหรับการสร้าง Embedding มีหลายวิธีเช่น Fixed-size Recursive และ Semantic',
    'Cosine Similarity เป็นวิธีวัดความคล้ายระหว่างสอง Vector โดยดูจากมุมระหว่าง Vector ค่า 1 หมายถึงทิศทางเดียวกัน นิยมใช้ในงาน NLP และ Information Retrieval',
    'Agent คือระบบ AI ที่สามารถตัดสินใจและใช้เครื่องมือได้ด้วยตัวเอง ต่างจาก Chatbot ที่ทำได้แค่ถาม-ตอบตาม script ที่กำหนดไว้',
    'Google ADK หรือ Agent Development Kit เป็นเฟรมเวิร์คสำหรับสร้าง AI Agent ด้วย Python รองรับ Multi-Agent และ Tool Calling ทำงานร่วมกับ Gemini ได้ดี',
]

random.shuffle(all_paragraphs)
selected = all_paragraphs[:8]

# สร้าง query เฉพาะตัว
all_queries = [
    'เทคนิคการค้นหาข้อมูลที่มีความหมายคล้ายกัน',
    'วิธีการแบ่งเอกสารเป็นส่วนย่อย',
    'การใช้ AI ตัดสินใจและเรียกใช้เครื่องมือ',
    'การแปลงข้อความเป็นตัวเลขเพื่อเปรียบเทียบ',
    'เทคนิคการสร้างคำตอบจากข้อมูลที่ค้นพบ',
]
random.shuffle(all_queries)
MY_QUERY = all_queries[0]

os.makedirs('homework_data', exist_ok=True)
for i, para in enumerate(selected):
    with open(f'homework_data/doc_{i+1}.txt', 'w', encoding='utf-8') as f:
        f.write(para)

print(f'✅ สร้างข้อมูลเฉพาะสำหรับ {STUDENT_ID}')
print(f'📁 จำนวนไฟล์: {len(selected)} ไฟล์')
print(f'🔍 Query เฉพาะตัว: "{MY_QUERY}"')
for i in range(len(selected)):
    print(f'  📄 doc_{i+1}.txt ({len(selected[i])} ตัวอักษร)')

---
## 🎯 ขั้นตอนที่ 1: Chunk + Embed + Search (3 คะแนน) — เฉลย

**สิ่งที่ต้องทำ:**
1. รวมข้อความจากทุกไฟล์ใน `homework_data/`
2. Chunk ด้วย `RecursiveCharacterTextSplitter` — `chunk_size=150`, `chunk_overlap=30`
3. สร้าง Embedding ด้วย `intfloat/multilingual-e5-large`
4. ค้นหาด้วย `MY_QUERY` + Cosine Similarity
5. เก็บลง Qdrant collection `f'hw_{STUDENT_ID}'`

In [ ]:
%%time
# ===== เฉลยขั้นตอนที่ 1 =====

# 1) อ่านไฟล์ทั้งหมดจาก homework_data/
all_text = ''
for fname in sorted(os.listdir('homework_data')):
    fpath = f'homework_data/{fname}'
    if os.path.isfile(fpath) and fname.endswith('.txt'):
        with open(fpath, 'r', encoding='utf-8') as f:
            all_text += '\n\n' + f.read()

print(f'📄 ข้อความรวม: {len(all_text)} ตัวอักษร')

# 2) Chunk ด้วย RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=30,
    separators=['\n\n', '\n', ' ', '']
)
chunks = splitter.split_text(all_text)
print(f'✂️ ได้ {len(chunks)} chunks')

# 3) สร้าง Embedding
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('intfloat/multilingual-e5-large')
passages = ['passage: ' + c for c in chunks]
chunk_embeddings = embed_model.encode(passages, show_progress_bar=True)
print(f'🔢 Embedding shape: {chunk_embeddings.shape}')

# 4) ค้นหาด้วย Cosine Similarity
query_emb = embed_model.encode(f'query: {MY_QUERY}')
scores = cosine_similarity([query_emb], chunk_embeddings)[0]
top_idx = np.argsort(scores)[::-1][:3]

print(f'\n🔍 Query: "{MY_QUERY}"')
print(f'📊 Top-3 Cosine Similarity:')
for rank, idx in enumerate(top_idx, 1):
    print(f'  {rank}. [score={scores[idx]:.4f}] {chunks[idx][:80]}...')

# 5) เก็บลง Qdrant
from qdrant_client import QdrantClient, models

qdrant = QdrantClient(':memory:')
COLLECTION = f'hw_{STUDENT_ID}'
VECTOR_SIZE = chunk_embeddings.shape[1]

qdrant.create_collection(
    collection_name=COLLECTION,
    vectors_config=models.VectorParams(size=VECTOR_SIZE, distance=models.Distance.COSINE)
)

points = [
    models.PointStruct(
        id=i,
        vector=chunk_embeddings[i].tolist(),
        payload={'text': chunks[i], 'chunk_id': i}
    )
    for i in range(len(chunks))
]
qdrant.upsert(collection_name=COLLECTION, points=points)
print(f'\n✅ Upsert {len(points)} chunks เข้า Qdrant collection "{COLLECTION}"')

# ค้นหาจาก Qdrant
q_emb = embed_model.encode(f'query: {MY_QUERY}')
qdrant_results = qdrant.query_points(
    collection_name=COLLECTION,
    query=q_emb.tolist(),
    limit=3
).points

print(f'\n📊 Top-3 จาก Qdrant:')
for i, r in enumerate(qdrant_results, 1):
    print(f'  {i}. [score={r.score:.4f}] {r.payload["text"][:80]}...')

# ✅ Self-check
assert len(chunks) > 0, '❌ ยังไม่ได้ chunk'
assert len(qdrant_results) == 3, '❌ ควรได้ top_k=3 จาก Qdrant'
print(f'\n✅ Step 1 passed: {len(chunks)} chunks, top score={qdrant_results[0].score:.4f}')

---
## 🎯 ขั้นตอนที่ 2: Agent + Custom Tool (3 คะแนน) — เฉลย

**ตัวอย่างเฉลย:** คำนวณ GPA จากคะแนน

> 💡 นักศึกษาแต่ละคนจะสร้าง tool ต่างกัน — ตรวจว่า:
> 1. ไม่ใช้ BMI (ซ้ำกับในคาบ)
> 2. มี docstring ครบถ้วน
> 3. Agent เรียก tool สำเร็จ

In [ ]:
# ===== เฉลยขั้นตอนที่ 2 =====

# ตั้งค่า API Key
import os
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
except Exception:
    os.environ['GOOGLE_API_KEY'] = input('🔑 วาง API Key: ')

from google.adk.agents import LlmAgent
from google.adk.tools import FunctionTool
from google.adk.runners import InMemoryRunner
from google.genai import types as genai_types

# ─── Custom Tool: คำนวณ GPA ───
def calculate_gpa(score: float, total: float) -> str:
    """คำนวณเกรดจากคะแนนสอบ

    คำนวณเกรดตามเกณฑ์มหาวิทยาลัย โดยรับคะแนนที่ได้และคะแนนเต็ม
    แล้วคืนค่าเกรดและ GPA ที่ได้

    Args:
        score: คะแนนที่ได้
        total: คะแนนเต็ม
    """
    pct = (score / total) * 100
    if pct >= 80: grade, gpa = 'A', 4.0
    elif pct >= 75: grade, gpa = 'B+', 3.5
    elif pct >= 70: grade, gpa = 'B', 3.0
    elif pct >= 65: grade, gpa = 'C+', 2.5
    elif pct >= 60: grade, gpa = 'C', 2.0
    elif pct >= 55: grade, gpa = 'D+', 1.5
    elif pct >= 50: grade, gpa = 'D', 1.0
    else: grade, gpa = 'F', 0.0
    return f'คะแนน {score}/{total} ({pct:.1f}%) → เกรด {grade} (GPA {gpa})'

tool = FunctionTool(calculate_gpa)

# ─── สร้าง Agent ───
my_agent = LlmAgent(
    name='grade_assistant',
    model='gemini-3.1-pro-preview',
    instruction='คุณเป็นผู้ช่วยคำนวณเกรด ใช้ tool calculate_gpa เมื่อผู้ใช้บอกคะแนน ตอบเป็นภาษาไทย',
    tools=[tool]
)

# ─── ทดสอบ ───
async def chat_with_agent(agent, message):
    runner = InMemoryRunner(agent=agent, app_name='homework')
    session = await runner.session_service.create_session(
        app_name='homework', user_id='student'
    )
    content = genai_types.Content(
        role='user', parts=[genai_types.Part(text=message)]
    )
    response_text = ''
    async for event in runner.run_async(
        user_id='student', session_id=session.id, new_message=content
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    response_text += part.text
    return response_text

# ถามคำถามที่ต้องใช้ tool
answer = await chat_with_agent(my_agent, 'ผมได้คะแนน 72 จาก 100 คะแนน ได้เกรดอะไรครับ')
print(f'🤖 Agent: {answer}')

# ✅ Self-check
assert answer is not None and len(answer) > 0, '❌ Agent ไม่ตอบ'
print(f'\n✅ Step 2 passed!')

### 📝 ตัวอย่างรายงาน Step 2

1. **Tool ทำอะไร?** — Tool `calculate_gpa` รับคะแนนที่ได้และคะแนนเต็ม แล้วคำนวณเปอร์เซ็นต์เพื่อให้เกรดและ GPA ตามเกณฑ์มหาวิทยาลัย
2. **Output** — Agent เรียก `calculate_gpa(score=72, total=100)` และตอบว่าได้เกรด B (GPA 3.0)
3. **ทำไม docstring สำคัญ?** — Agent ใช้ docstring เพื่อตัดสินใจว่าจะเรียก tool ไหน ถ้า docstring ไม่ชัดเจน Agent อาจเลือก tool ผิดหรือไม่เรียก tool เลย

---
## 🎯 ขั้นตอนที่ 3: RAG Agent + วัดคุณภาพ (4 คะแนน) — เฉลย

In [ ]:
# ===== เฉลยขั้นตอนที่ 3 =====

# ─── A) สร้าง RAG Tool ───
def search_knowledge(query: str) -> str:
    """ค้นหาข้อมูลจากฐานความรู้ที่จัดเก็บใน Qdrant
    ใช้เมื่อต้องการหาข้อมูลเกี่ยวกับ AI, Machine Learning, NLP

    Args:
        query: คำถามหรือหัวข้อที่ต้องการค้นหา
    """
    q_emb = embed_model.encode(f'query: {query}')
    results = qdrant.query_points(
        collection_name=COLLECTION,
        query=q_emb.tolist(),
        limit=3
    ).points
    
    if not results:
        return 'ไม่พบข้อมูลที่เกี่ยวข้อง'
    
    context = '\n\n'.join([
        f'[{i+1}] (score={r.score:.4f}) {r.payload["text"]}'
        for i, r in enumerate(results)
    ])
    return f'ผลการค้นหา {len(results)} รายการ:\n{context}'

# ─── B) สร้าง RAG Agent ───
rag_agent = LlmAgent(
    name='rag_assistant',
    model='gemini-3.1-pro-preview',
    instruction="""คุณเป็น AI ที่ตอบคำถามจากฐานความรู้

กฎ:
1. ใช้ tool search_knowledge ค้นหาข้อมูลก่อนตอบเสมอ
2. ตอบจากข้อมูลที่ค้นพบเท่านั้น ห้ามแต่งเอง
3. ถ้าไม่พบข้อมูล ให้บอกตรงๆ ว่าไม่มีข้อมูล
4. ตอบเป็นภาษาไทย สั้นกระชับ""",
    tools=[FunctionTool(search_knowledge)]
)

print('✅ RAG Agent + RAG Tool พร้อมใช้งาน')

In [ ]:
# ─── C) ถาม 3 คำถาม ───
questions = [
    MY_QUERY,  # query เฉพาะตัว
    'Embedding คืออะไร?',
    'ทำไม RAG ถึงสำคัญ?'
]

rag_answers = []
for q in questions:
    ans = await chat_with_agent(rag_agent, q)
    rag_answers.append({'question': q, 'answer': ans})
    print(f'\n{"="*50}')
    print(f'❓ {q}')
    print(f'🤖 {ans}')

# ─── D) LLM-as-Judge ───
from google import genai
judge_client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])

JUDGE_PROMPT = '''คุณเป็นผู้ตรวจคุณภาพคำตอบ AI

คำถาม: {question}
คำตอบ: {answer}

ให้คะแนน 1-5 ตามเกณฑ์:
- 5 = ถูกต้อง ครบถ้วน อธิบายชัดเจน
- 4 = ถูกต้อง แต่ขาดรายละเอียดบางส่วน
- 3 = ถูกบางส่วน มีข้อผิดพลาดเล็กน้อย
- 2 = ตอบไม่ตรงประเด็น หรือผิดหลายจุด
- 1 = ผิดทั้งหมด หรือไม่ตอบ

ตอบ JSON: {{"score": 0, "reason": "..."}}
'''

print(f'\n{"="*50}')
print('📊 LLM-as-Judge Results:')
for qa in rag_answers:
    prompt = JUDGE_PROMPT.format(question=qa['question'], answer=qa['answer'])
    resp = judge_client.models.generate_content(
        model='gemini-3.1-pro-preview', contents=prompt,
        config=genai.types.GenerateContentConfig(temperature=0.1, response_mime_type='application/json')
    )
    result = json.loads(resp.text)
    print(f'  ❓ {qa["question"][:40]}... → ⭐ {result["score"]}/5 — {result["reason"]}')

# ✅ Self-check
assert len(rag_answers) == 3, '❌ ต้องตอบ 3 คำถาม'
print(f'\n✅ Step 3 passed: ตอบครบ 3 ข้อ + LLM-as-Judge เสร็จ!')

### 📝 ตัวอย่างรายงาน Step 3

**Agent ตัดสินใจค้นหาจาก Qdrant อย่างไร?**

Agent อ่านคำถามแล้ว ตรวจสอบว่ามี tool `search_knowledge` ที่เหมาะกับคำถามหรือไม่
โดย Agent อ่าน docstring ของ tool แล้วพบว่า tool นี้ "ค้นหาข้อมูลเกี่ยวกับ AI, Machine Learning, NLP" 
ซึ่งตรงกับคำถาม จึงตัดสินใจเรียก tool พร้อมส่ง query ที่เหมาะสม
จากนั้น Agent นำผลลัพธ์จาก Qdrant มาสร้างคำตอบเป็นภาษาธรรมชาติ

## 📊 เกณฑ์การให้คะแนน

| ขั้นตอน | คะแนน | เกณฑ์ |
|---------|:-----:|------|
| 1. Chunk + Embed + Search | 3 | Pipeline ทำงานได้, ผล Qdrant ถูกต้อง |
| 2. Agent + Custom Tool | 3 | Tool ทำงาน, Agent เรียกใช้ได้, อธิบาย docstring |
| 3. RAG Agent + Judge | 4 | RAG Agent ตอบครบ 3 ข้อ, LLM-as-Judge ให้คะแนน, อธิบาย |
| **รวม** | **10** | |

---
## ✅ ตรวจสอบคำตอบ

Run cell ด้านล่างเพื่อสร้าง **Verification Code**

In [ ]:
# ===== ห้ามแก้ไข cell นี้ =====
verify_hash = hashlib.sha256(f'{STUDENT_ID}_4hr_hw'.encode()).hexdigest()[:12]
print('=' * 50)
print(f'👤 ชื่อ-นามสกุล: {STUDENT_NAME}')
print(f'🎓 รหัสนักศึกษา: {STUDENT_ID}')
print(f'📱 เบอร์โทร: {PHONE}')
print(f'💬 LINE ID: {LINE_ID}')
print(f'🔑 Verification Code: {verify_hash}')
print(f'📅 ส่งก่อน: 24 มี.ค. 2569 23:59 น.')
print('=' * 50)
print()
print('📋 Checklist ก่อนส่ง:')
print('  [ ] กรอกข้อมูลส่วนตัวครบถ้วน')
print('  [ ] ขั้นตอนที่ 1: Chunk + Embed + Qdrant ทำงาน')
print('  [ ] ขั้นตอนที่ 2: Agent + Tool ทำงาน')
print('  [ ] ขั้นตอนที่ 3: RAG Agent ตอบ 3 ข้อ + LLM-as-Judge')
print('  [ ] ทุก cell run แล้วมีผลลัพธ์')